# Panel Data Construction
## Brazilian Chamber of Deputies Speech Panel

This notebook constructs a panel dataset from the preprocessed corpus, adding:

1. **Party Affiliation History** - Time-varying party membership
2. **Ideological Scores** - Power & Zucco estimates (EST, CAT)
3. **Covariates** - Government loyalty, legislative activity, tenure
4. **Embeddings & Topics** - SBERT embeddings and LDA topic vectors

# Section 1: Setup & Configuration

In [36]:
import os
import glob
import bisect
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

warnings.filterwarnings('ignore')
print("Imports complete.")

Imports complete.


In [19]:
@dataclass
class PanelConfig:
    corpus_path: str = '../data/processed/data_corpus.parquet'
    embeddings_path: str = '../data/processed/data_embeddings.parquet'
    lda_path: str = '../data/processed/data_lda_topics.parquet'
    deputy_history_path: str = '../data/raw/deputies/deputy_migrations.csv'
    deputy_seniority_path: str = '../data/raw/deputies/deputy_tenure.csv'
    ideology_path: str = '../data/raw/parties/power_zucco_point_estimates.csv'
    votes_path: str = '../data/raw/scrape_votes.parquet'
    propositions_dir: str = '../data/raw/bills_proposed/'
    output_dir: str = '../data/processed/'
    left_threshold: float = -0.25
    right_threshold: float = 0.25
    gov_loyalty_window: str = '365D'

CFG = PanelConfig()
print(f"Corpus: {CFG.corpus_path}")
print(f"Seniority: {CFG.deputy_seniority_path}")

Corpus: ../data/processed/data_corpus.parquet
Seniority: ../data/raw/perfil_deputados/senioridade_deputados.csv


# Section 2: Load Corpus

In [20]:
print("Loading preprocessed corpus...")
df_corpus = pd.read_parquet(CFG.corpus_path)

keep_cols = ['dataHoraInicio', 'tipoDiscurso', 'keywords', 'sumario',
             'deputado_id', 'idLegislatura', 'is_outlier',
             'text_level_1', 'text_level_2', 'text_level_3', 'text_level_4']
keep_cols = [c for c in keep_cols if c in df_corpus.columns]
df = df_corpus[keep_cols].copy()

df['deputado_id'] = df['deputado_id'].astype(str)
df['dataHoraInicio'] = pd.to_datetime(df['dataHoraInicio'], utc=True)
df = df.dropna(subset=['deputado_id', 'dataHoraInicio'])

n_initial = len(df)
print(f"Loaded {n_initial:,} speeches from {df['deputado_id'].nunique():,} deputies")

Loading preprocessed corpus...
Loaded 372,401 speeches from 1,667 deputies


# Section 3: Party Affiliation

In [21]:
print("Stage 1: Party Affiliation")
print("="*70)

df_hist = pd.read_csv(CFG.deputy_history_path)
if df_hist.columns.duplicated().any():
    df_hist = df_hist.loc[:, ~df_hist.columns.duplicated()]

df_hist['idPartido'] = df_hist['uriPartido'].str.split('/').str[-1]
df_hist['deputado_id'] = df_hist['deputado_id'].astype(str)
df_hist['dataHora'] = pd.to_datetime(df_hist['dataHora'], utc=True)
df_hist = df_hist.sort_values(['deputado_id', 'dataHora'])

print(f"Loaded history for {df_hist['deputado_id'].nunique():,} deputies")

Stage 1: Party Affiliation
Loaded history for 1,788 deputies


In [22]:
# Synthetic events for non-switchers
is_switch = df_hist['descricaoStatus'] == 'Alteração de partido'
switcher_ids = df_hist.loc[is_switch, 'deputado_id'].unique()
never_switched = df_hist.loc[~df_hist['deputado_id'].isin(switcher_ids), 'deputado_id'].unique()

synthetic_rows = []
for did in never_switched:
    first_row = df_hist[df_hist['deputado_id'] == did].iloc[0].copy()
    first_row['descricaoStatus'] = 'Alteração de partido'
    synthetic_rows.append(first_row)

if synthetic_rows:
    df_hist = pd.concat([df_hist, pd.DataFrame(synthetic_rows)], ignore_index=True)
    df_hist = df_hist.sort_values(['deputado_id', 'dataHora'])

print(f"Added {len(synthetic_rows)} synthetic events for non-switchers")

Added 855 synthetic events for non-switchers


In [23]:
# Build party-phase lookup
is_switch = df_hist['descricaoStatus'] == 'Alteração de partido'
first_switch_idx = df_hist[is_switch].groupby('deputado_id', sort=False)['dataHora'].idxmin()
keep_as_well = first_switch_idx - 1
mask = is_switch | df_hist.index.isin(keep_as_well.dropna())

hist = df_hist[mask][['deputado_id', 'idPartido', 'siglaPartido', 'dataHora']].copy()
hist = hist.sort_values(['deputado_id', 'dataHora'])

def build_changes_tuple(hist):
    def to_tuple(group):
        return tuple((r.dataHora, r.idPartido, r.siglaPartido) for _, r in group.iterrows())
    return hist.groupby('deputado_id', sort=False).apply(to_tuple, include_groups=False).rename('changes_tuple').reset_index()

changes_df = build_changes_tuple(hist)
print(f"Built lookup tuples for {len(changes_df):,} deputies")

Built lookup tuples for 1,788 deputies


In [24]:
# Apply party lookup
df = df.merge(changes_df, on='deputado_id', how='left')

def party_from_tuple(changes_tuple, speech_time):
    if pd.isna(changes_tuple) or changes_tuple is None:
        return pd.NA, pd.NA, pd.NA
    times = [t for t, _, _ in changes_tuple]
    pos = max(0, bisect.bisect_right(times, speech_time) - 1)
    _, idP, sigP = changes_tuple[pos]
    return idP, sigP, pos

df[['idPartido', 'siglaPartido', 'party_change_count']] = df.progress_apply(
    lambda r: party_from_tuple(r.changes_tuple, r.dataHoraInicio), axis=1, result_type='expand')

# Drop the tuple column - no longer needed
df = df.drop(columns=['changes_tuple'])

print(f"Parties assigned. Unique parties: {df['siglaPartido'].nunique()}")
print(f"Rows: {len(df):,} (should be {n_initial:,})")

100%|██████████| 372401/372401 [00:05<00:00, 64122.12it/s] 

Parties assigned. Unique parties: 51
Rows: 372,401 (should be 372,401)


# Section 4: Ideology

In [25]:
print("\nStage 2: Ideology")
print("="*70)

df_ideology = pd.read_csv(CFG.ideology_path)
if 'id' in df_ideology.columns:
    df_ideology = df_ideology.rename(columns={'id': 'idPartido'})

value_cols = sorted([c for c in df_ideology.columns if c.isdigit()], key=int)
long = (df_ideology.melt(id_vars=['idPartido', 'type'], value_vars=value_cols, var_name='year_start', value_name='value')
        .assign(year_start=lambda d: d['year_start'].astype(int)))
long = long.pivot_table(index=['idPartido', 'year_start'], columns='type', values='value', aggfunc='first').reset_index()
changepoints = sorted(int(c) for c in value_cols)

print(f"Ideology table: {len(long):,} party-year combinations")
print(f"Changepoints: {changepoints}")


Stage 2: Ideology
Ideology table: 297 party-year combinations
Changepoints: [1990, 1993, 1997, 2001, 2005, 2009, 2013, 2017, 2021]


In [26]:
def map_year_to_changepoint(yr, cps):
    return cps[max(0, bisect.bisect_right(cps, yr) - 1)]

df['year'] = df['dataHoraInicio'].dt.year
df['year_start'] = df['year'].apply(lambda y: map_year_to_changepoint(y, changepoints))
df['idPartido'] = df['idPartido'].astype(str)
long['idPartido'] = long['idPartido'].astype(str)

df = df.merge(long, on=['idPartido', 'year_start'], how='left')

def est_to_cat(est):
    if pd.isna(est): return pd.NA
    if est < CFG.left_threshold: return 0
    if est <= CFG.right_threshold: return 1
    return 2

df['CAT'] = df['EST'].apply(est_to_cat).astype('Int64')
df = df.drop(columns=['year', 'year_start'])

print(f"Ideology assigned. Distribution:")
print(df['CAT'].value_counts().sort_index())
print(f"Rows: {len(df):,} (should be {n_initial:,})")

Ideology assigned. Distribution:
CAT
0    147871
1     91161
2    126519
Name: count, dtype: Int64
Rows: 372,401 (should be 372,401)


# Section 5: Covariates

In [27]:
print("\nStage 3: Covariates")
print("="*70)

# Government loyalty
print("\nComputing government loyalty index...")
df_gov_index = None

if os.path.exists(CFG.votes_path):
    df_votes = pd.read_parquet(CFG.votes_path)
    df_votes['deputado_id'] = df_votes['deputado_id'].astype(str)
    df_votes['dataVotacao'] = pd.to_datetime(df_votes['dataVotacao']).dt.tz_localize(None)
    
    df_conflict = df_votes[
        (df_votes['gov_orient'].isin(['Sim', 'Não'])) &
        (df_votes['oposicao_orient'].isin(['Sim', 'Não'])) &
        (df_votes['gov_orient'] != df_votes['oposicao_orient'])].copy()
    df_conflict['match_gov'] = (df_conflict['voto'] == df_conflict['gov_orient']).astype(int)
    
    df_gov_index = (df_conflict.sort_values('dataVotacao').set_index('dataVotacao')
                    .groupby('deputado_id')['match_gov'].rolling(CFG.gov_loyalty_window).mean()
                    .reset_index(name='gov_loyalty_12m'))
    df_gov_index['dataVotacao'] = pd.to_datetime(df_gov_index['dataVotacao']).dt.tz_localize(None)
    print(f"Computed loyalty for {df_gov_index['deputado_id'].nunique():,} deputies")
else:
    print("Votes file not found")


Stage 3: Covariates

Computing government loyalty index...
Computed loyalty for 1,053 deputies


In [28]:
# Proposition activity - aggregate to ONE ROW per (deputado_id, ano)
print("\nComputing legislative activity...")
df_prop_agg = None

author_pattern = os.path.join(CFG.propositions_dir, 'proposicoesAutores-*.csv')
author_files = glob.glob(author_pattern)

if author_files:
    annual_list = []
    for f in author_files:
        ano = int(os.path.basename(f).split('-')[1].split('.')[0])
        tmp = pd.read_csv(f, sep=';', usecols=['idDeputadoAutor'])
        tmp = tmp.dropna(subset=['idDeputadoAutor'])
        count = tmp.groupby('idDeputadoAutor').size().reset_index(name='prop_count')
        count['ano'] = ano
        annual_list.append(count)
    
    df_prop_agg = pd.concat(annual_list).rename(columns={'idDeputadoAutor': 'deputado_id'})
    df_prop_agg['deputado_id'] = df_prop_agg['deputado_id'].astype(int).astype(str)  # float -> int -> str
    df_prop_agg = df_prop_agg.groupby(['deputado_id', 'ano'], as_index=False)['prop_count'].sum()
    df_prop_agg = df_prop_agg.rename(columns={'prop_count': 'prop_activity_annual'})
    
    print(f"Loaded activity for {df_prop_agg['deputado_id'].nunique():,} deputies")
    print(f"Unique (deputado_id, ano) rows: {len(df_prop_agg):,}")
else:
    print("No proposition files found")


Computing legislative activity...
Loaded activity for 2,241 deputies
Unique (deputado_id, ano) rows: 13,384


In [29]:
# Merge covariates
print("\nMerging covariates...")

# Prepare naive datetime for merge_asof
df['dataHoraInicio_naive'] = df['dataHoraInicio'].dt.tz_localize(None)
df['deputado_id_int'] = pd.to_numeric(df['deputado_id'], errors='coerce').fillna(-1).astype('int64')
df = df.sort_values('dataHoraInicio_naive')

# Government loyalty (merge_asof - backward lookup)
if df_gov_index is not None:
    df_gov_index['deputado_id_int'] = pd.to_numeric(df_gov_index['deputado_id'], errors='coerce').fillna(-1).astype('int64')
    df_gov_index = df_gov_index.sort_values('dataVotacao')
    
    df = pd.merge_asof(
        df,
        df_gov_index[['deputado_id_int', 'dataVotacao', 'gov_loyalty_12m']],
        by='deputado_id_int',
        left_on='dataHoraInicio_naive',
        right_on='dataVotacao',
        direction='backward'
    )
    df['gov_loyalty_12m'] = df['gov_loyalty_12m'].fillna(0.5)
    df = df.drop(columns=['dataVotacao'], errors='ignore')
    print(f"  Merged government loyalty. Rows: {len(df):,}")

# Proposition activity (simple merge on deputado_id + year)
if df_prop_agg is not None:
    df['ano'] = df['dataHoraInicio'].dt.year
    df = df.merge(df_prop_agg, on=['deputado_id', 'ano'], how='left')
    df['prop_activity_annual'] = df['prop_activity_annual'].fillna(0).astype(int)
    df = df.drop(columns=['ano'], errors='ignore')
    print(f"  Merged proposition activity. Rows: {len(df):,}")

df = df.drop(columns=['deputado_id_int'], errors='ignore')
print(f"\nAfter covariates: {len(df):,} rows (should be {n_initial:,})")


Merging covariates...
  Merged government loyalty. Rows: 372,401
  Merged proposition activity. Rows: 372,401

After covariates: 372,401 rows (should be 372,401)


In [30]:
# Tenure columns
print("\nComputing tenure...")

# Load seniority
df_seniority = pd.read_csv(CFG.deputy_seniority_path)
df_seniority['deputado_id'] = df_seniority['deputado_id'].astype(str)
df_seniority['data_inicio_carreira'] = pd.to_datetime(df_seniority['data_inicio_carreira']).dt.tz_localize(None)
print(f"  Loaded seniority for {len(df_seniority):,} deputies")

# Some deputies have speeches BEFORE their recorded career start (data issue)
# Use min(recorded_career_start, first_speech_date) as true career start
first_speech = df.groupby('deputado_id')['dataHoraInicio_naive'].min().reset_index()
first_speech.columns = ['deputado_id', 'first_speech_date']

df_seniority = df_seniority.merge(first_speech, on='deputado_id', how='left')
df_seniority['data_inicio_carreira'] = df_seniority[['data_inicio_carreira', 'first_speech_date']].min(axis=1)
df_seniority = df_seniority.drop(columns=['first_speech_date'])

# Merge career start
df = df.merge(df_seniority[['deputado_id', 'data_inicio_carreira']], on='deputado_id', how='left')

# Party changes for party tenure
df_party_changes = df_hist[df_hist['descricaoStatus'] == 'Alteração de partido'][['deputado_id', 'dataHora']].copy()
df_party_changes.columns = ['deputado_id', 'data_ultimo_partido']
df_party_changes['data_ultimo_partido'] = df_party_changes['data_ultimo_partido'].dt.tz_localize(None)
df_party_changes = df_party_changes.sort_values(['deputado_id', 'data_ultimo_partido'])
df_party_changes['deputado_id_int'] = pd.to_numeric(df_party_changes['deputado_id'], errors='coerce').fillna(-1).astype('int64')

df['deputado_id_int'] = pd.to_numeric(df['deputado_id'], errors='coerce').fillna(-1).astype('int64')
df = df.sort_values('dataHoraInicio_naive')

# Merge last party change BEFORE each speech
df = pd.merge_asof(
    df,
    df_party_changes[['deputado_id_int', 'data_ultimo_partido']].sort_values('data_ultimo_partido'),
    by='deputado_id_int',
    left_on='dataHoraInicio_naive',
    right_on='data_ultimo_partido',
    direction='backward'
)

# If never changed party, use career start
df['data_ultimo_partido'] = df['data_ultimo_partido'].fillna(df['data_inicio_carreira'])

# Calculate tenure
df['career_tenure_years'] = ((df['dataHoraInicio_naive'] - df['data_inicio_carreira']).dt.days / 365.25).round(2)
df['party_tenure_months'] = ((df['dataHoraInicio_naive'] - df['data_ultimo_partido']).dt.days / 30.44).round(1)

# Cleanup
df = df.drop(columns=['dataHoraInicio_naive', 'deputado_id_int', 'data_inicio_carreira', 'data_ultimo_partido'], errors='ignore')

print(f"  Career tenure: {df['career_tenure_years'].min():.1f} - {df['career_tenure_years'].max():.1f} years")
print(f"  Party tenure: {df['party_tenure_months'].min():.1f} - {df['party_tenure_months'].max():.1f} months")

# Verify
n_neg = ((df['career_tenure_years'] < 0).sum() + (df['party_tenure_months'] < 0).sum())
if n_neg > 0:
    print(f"  WARNING: {n_neg} negative tenure values!")
else:
    print(f"  ✓ All tenure values >= 0")

print(f"\nAfter tenure: {len(df):,} rows (should be {n_initial:,})")


Computing tenure...
  Loaded seniority for 1,788 deputies
  Career tenure: 0.0 - 51.4 years
  Party tenure: 0.0 - 517.3 months
  ✓ All tenure values >= 0

After tenure: 372,401 rows (should be 372,401)


# Section 6: Embeddings & Topics

In [31]:
print("\nStage 4: Embeddings & Topics")
print("="*70)

# Embeddings - LEFT JOIN to keep all speeches
if os.path.exists(CFG.embeddings_path):
    df_emb = pd.read_parquet(CFG.embeddings_path)
    df_emb['deputado_id'] = df_emb['deputado_id'].astype(str)
    df_emb['dataHoraInicio'] = pd.to_datetime(df_emb['dataHoraInicio'], utc=True)
    
    # Deduplicate embeddings to ensure 1:1 merge
    df_emb = df_emb.drop_duplicates(subset=['deputado_id', 'dataHoraInicio'], keep='first')
    
    df = df.merge(
        df_emb[['deputado_id', 'dataHoraInicio', 'embedding']],
        on=['deputado_id', 'dataHoraInicio'],
        how='left'
    )
    print(f"Embeddings: {df['embedding'].notna().sum():,} / {len(df):,} ({100*df['embedding'].notna().mean():.1f}%)")
else:
    print("Embeddings file not found")

print(f"After embeddings: {len(df):,} rows (should be {n_initial:,})")


Stage 4: Embeddings & Topics
Embeddings: 365,551 / 372,401 (98.2%)
After embeddings: 372,401 rows (should be 372,401)


In [32]:
# LDA Topics - LEFT JOIN to keep all speeches
if os.path.exists(CFG.lda_path):
    df_lda = pd.read_parquet(CFG.lda_path)
    df_lda['deputado_id'] = df_lda['deputado_id'].astype(str)
    df_lda['dataHoraInicio'] = pd.to_datetime(df_lda['dataHoraInicio'], utc=True)
    
    # Deduplicate LDA to ensure 1:1 merge
    df_lda = df_lda.drop_duplicates(subset=['deputado_id', 'dataHoraInicio'], keep='first')
    
    df = df.merge(
        df_lda[['deputado_id', 'dataHoraInicio', 'topic_id', 'topic_vector']],
        on=['deputado_id', 'dataHoraInicio'],
        how='left'
    )
    print(f"Topics: {df['topic_id'].notna().sum():,} / {len(df):,} ({100*df['topic_id'].notna().mean():.1f}%)")
else:
    print("LDA file not found")

print(f"After LDA: {len(df):,} rows (should be {n_initial:,})")

Topics: 372,401 / 372,401 (100.0%)
After LDA: 372,401 rows (should be 372,401)


# Section 7: Save

In [33]:
# Final verification
print("\nFinal verification...")
print(f"  Expected rows: {n_initial:,}")
print(f"  Actual rows: {len(df):,}")

if len(df) != n_initial:
    print(f"  WARNING: Row count mismatch! Difference: {len(df) - n_initial:,}")
    # Find and remove duplicates
    print(f"  Removing duplicates...")
    df = df.drop_duplicates(subset=['deputado_id', 'dataHoraInicio'], keep='first')
    print(f"  After dedup: {len(df):,}")
else:
    print(f"  ✓ Row count matches")


Final verification...
  Expected rows: 372,401
  Actual rows: 372,401
  ✓ Row count matches


In [34]:
# Sort and save
df = df.sort_values(['deputado_id', 'dataHoraInicio']).reset_index(drop=True)

output_path = os.path.join(CFG.output_dir, 'data_panel.parquet')
df.to_parquet(output_path, index=False, compression='brotli')

print(f"\nSaved: {output_path}")
print(f"Shape: {df.shape}")
print(f"Size: {os.path.getsize(output_path) / 1e6:.1f} MB")


Saved: ../data/processed/data_panel.parquet
Shape: (372401, 24)
Size: 1511.1 MB


In [35]:
# Summary report
print("\n" + "="*80)
print("PANEL CONSTRUCTION REPORT")
print("="*80)

print(f"\nSAMPLE")
print(f"  Speeches: {len(df):,}")
print(f"  Deputies: {df['deputado_id'].nunique():,}")
print(f"  Parties: {df['siglaPartido'].nunique():,}")
print(f"  Date range: {df['dataHoraInicio'].min().date()} to {df['dataHoraInicio'].max().date()}")

print(f"\nPARTY SWITCHING")
switchers = df[df['party_change_count'] > 0]['deputado_id'].nunique()
print(f"  Deputies who switched: {switchers:,}")
print(f"  Max switches by one deputy: {df['party_change_count'].max()}")

print(f"\nIDEOLOGY")
for cat, label in [(0, 'Left'), (1, 'Center'), (2, 'Right')]:
    n = (df['CAT'] == cat).sum()
    pct = 100 * n / len(df)
    print(f"  {label}: {n:,} ({pct:.1f}%)")

print(f"\nCOVARIATES")
if 'gov_loyalty_12m' in df.columns:
    print(f"  Gov loyalty: mean={df['gov_loyalty_12m'].mean():.3f}, std={df['gov_loyalty_12m'].std():.3f}")
if 'prop_activity_annual' in df.columns:
    print(f"  Propositions/year: mean={df['prop_activity_annual'].mean():.1f}, max={df['prop_activity_annual'].max()}")
if 'career_tenure_years' in df.columns:
    print(f"  Career tenure: mean={df['career_tenure_years'].mean():.1f} years")
if 'party_tenure_months' in df.columns:
    print(f"  Party tenure: mean={df['party_tenure_months'].mean():.1f} months")

print(f"\nEMBEDDINGS & TOPICS")
if 'embedding' in df.columns:
    print(f"  With embeddings: {df['embedding'].notna().sum():,} ({100*df['embedding'].notna().mean():.1f}%)")
if 'topic_id' in df.columns:
    print(f"  With topics: {df['topic_id'].notna().sum():,} ({100*df['topic_id'].notna().mean():.1f}%)")

print("\n" + "="*80)
print("COMPLETE")
print("="*80)


PANEL CONSTRUCTION REPORT

SAMPLE
  Speeches: 372,401
  Deputies: 1,667
  Parties: 51
  Date range: 2003-02-01 to 2025-09-17

PARTY SWITCHING
  Deputies who switched: 783
  Max switches by one deputy: 8

IDEOLOGY
  Left: 147,871 (39.7%)
  Center: 91,161 (24.5%)
  Right: 126,519 (34.0%)

COVARIATES
  Gov loyalty: mean=0.488, std=0.194
  Propositions/year: mean=0.0, max=0
  Career tenure: mean=9.7 years
  Party tenure: mean=63.0 months

EMBEDDINGS & TOPICS
  With embeddings: 365,551 (98.2%)
  With topics: 372,401 (100.0%)

COMPLETE
